# Inspect Soil Moisture Target

QA/QC of the combined soil moisture calibration target written by
`targets/som.py`. The target emits TWO output files derived from a
single base name:

- `soil_moisture_targets_monthly.nc` (+ `_nn_filled.nc`) — monthly
  cadence, **per-calendar-month** [0, 1] normalization.
- `soil_moisture_targets_annual.nc` (+ `_nn_filled.nc`) — annual
  cadence (monthly→annual mean per source), **whole-period** [0, 1]
  normalization.

Up to four sources contribute on a shared fabric; each is independently
normalized and then combined with NaN-aware min / max:

- MERRA-2 `GWETTOP` (dimensionless plant-available wetness, 0–5 cm).
- NCEP/NCAR `soilw_0_10cm` (m³/m³ VWC despite kg/m² mislabel — recipes §4).
- NLDAS-2 MOSAIC `SoilM_0_10cm` (kg/m² in 0–10 cm).
- NLDAS-2 NOAH `SoilM_0_10cm` (kg/m² in 0–10 cm).

**For the gfv2 fabric, `ncep_ncar` is excluded** (per the
`project_gfv2_coarse_grid_exclusions` memory: its T62 Gaussian grid
≈ ~210 km is too coarse to resolve intermountain-west altitude
gradients). The default gfv2 target is therefore a 3-source bound.

This notebook checks BOTH variants in sequence — section 1 (monthly)
covers the per-calendar-month climatology, the per-time coverage at
monthly cadence, and a representative monthly map / time series.
Section 2 (annual) covers the per-year CONUS-mean envelope where the
interannual drought / wet-year signal lives.

Companion to:

- `inspect_aggregated_soil_moisture.ipynb` — per-source HRU aggregates
  before normalization + multi-source combination.

## Conventions

- HRU dim name follows `fabric.id_col` (gfv2: `nat_hru_id`).
- Bound units are dimensionless `1`.
- For the MONTHLY variant, the per-calendar-month normalization means
  the grand mean of the bound across (HRU, year) is NOT generally 0.5
  — that would only hold for symmetric distributions. The expected
  diagnostic is the climatological cycle (mean per calendar month
  across years) showing seasonal patterns in cross-source spread.
- For the ANNUAL variant, the whole-period normalization means the
  CONUS-mean trace should show clear interannual signal — wet years
  near the high end of [0, 1], dry years near the low end.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

from _helpers import (
    area_weighted_mean,
    area_weighted_series,
    load_fabric,
    load_project_paths,
    lookup_hrus_by_points,
    n_sources_per_time,
    nan_hru_count,
    open_target_nc,
    plot_categorical_choropleth,
    plot_hru_choropleth,
    plot_nan_hrus,
    save_figure,
    select_month,
)

# Edit me to point at a real project directory:
PROJECT_DIR = Path(
    "/caldera/hovenweep/projects/usgs/water/impd/nhgf/gfv2-spatial-targets"
)

# Set True (and re-run) to populate docs/figures/targets/<project>/*.png
import _helpers
_helpers.SAVE_FIGURES = True
_helpers.PROJECT = PROJECT_DIR.name

TARGET = "soil_moisture"

# Monthly-variant snapshot:
TARGET_YEAR = 2005
TARGET_MONTH = 4   # April — spring soil-moisture peak in much of CONUS
TARGET_TIME_MONTHLY = f"{TARGET_YEAR}-{TARGET_MONTH:02d}-01"
TARGET_TIME_ANNUAL = f"{TARGET_YEAR}-01-01"

REPRESENTATIVE_POINTS = {
    "Iowa cropland (root-zone storage)": (-93.6, 42.0),
    "Southern Appalachians (Eastern forest)": (-83.5, 35.5),
    "Phoenix metro (arid SW)": (-112.1, 33.4),
    "Olympic Peninsula (PNW conifer rainforest)": (-123.5, 47.8),
}

project_dir, datastore_dir, fabric_cfg = load_project_paths(PROJECT_DIR)
fabric = load_fabric(fabric_cfg)
id_dim = fabric_cfg["id_col"]

print(f"Project: {project_dir}")
print(f"Datastore: {datastore_dir}")
fabric_path = fabric_cfg['path']
print(f"Fabric: {fabric_path} ({len(fabric)} HRUs, id_col={id_dim!r})")
print(f"Monthly snapshot: {TARGET_TIME_MONTHLY}")
print(f"Annual snapshot:  {TARGET_TIME_ANNUAL}")

## Open both variants

The build emits two output NCs per variant (raw + NN-filled). Each
variant has its own per-calendar-month / whole-period normalization
applied independently; this notebook loads all four files (when
present) and sections below switch between them.

In [ ]:
targets_dir = project_dir / "targets"

variants = {}
for variant in ("monthly", "annual"):
    raw_p = targets_dir / f"soil_moisture_targets_{variant}.nc"
    nn_p = targets_dir / f"soil_moisture_targets_{variant}_nn_filled.nc"
    if not raw_p.exists():
        print(f"SKIP: {raw_p.name} not found.")
        continue
    raw = open_target_nc(raw_p)
    filled = open_target_nc(nn_p) if nn_p.exists() else None
    variants[variant] = (raw, filled)
    print(f"Loaded {variant}: {raw_p.name} ({raw_p.stat().st_size / 1e6:.1f} MB)" + (
        f"  + NN-filled ({nn_p.stat().st_size / 1e6:.1f} MB)" if filled is not None else ""
    ))

if not variants:
    print("No soil_moisture target files found. Run "
          "`pixi run run-som -- --project-dir <project>` first.")
    raise SystemExit

ds_monthly_raw = variants.get("monthly", (None, None))[0]
ds_monthly_filled = variants.get("monthly", (None, None))[1]
ds_annual_raw = variants.get("annual", (None, None))[0]
ds_annual_filled = variants.get("annual", (None, None))[1]

## Schema — monthly variant

In [ ]:
if ds_monthly_raw is None:
    print("SKIP: monthly variant not loaded.")
else:
    print(ds_monthly_raw)
    print()
    print("=== Global attrs ===")
    for k, v in ds_monthly_raw.attrs.items():
        print(f"  {k:<22} {v}")
    print()
    print("=== Per-variable units ===")
    for v in ("lower_bound", "upper_bound", "n_sources"):
        print(f"  {v:<14} units={ds_monthly_raw[v].attrs.get('units', '?')!r}  long_name={ds_monthly_raw[v].attrs.get('long_name', '')!r}")

## Per-time coverage — monthly variant

Per-month HRU count at each `n_sources` flag value. With gfv2's
3-source default (merra2 + nldas_mosaic + nldas_noah), expect `n=3`
across the interior CONUS for every month; `n=2` or `n=1` at fabric
edges where one source's grid doesn't reach.

In [ ]:
if ds_monthly_raw is None:
    print("SKIP")
else:
    cov_m = n_sources_per_time(ds_monthly_raw)
    print(cov_m.describe().T[["mean", "std", "min", "max"]])

    fig, ax = plt.subplots(figsize=(11, 4))
    cov_m.plot(ax=ax)
    ax.set_xlabel("Time")
    ax.set_ylabel("HRU count")
    ax.set_title(f"Per-month n_sources distribution — {TARGET} monthly")
    ax.legend(title="flag value", loc="center left", bbox_to_anchor=(1.0, 0.5))
    plt.tight_layout()
    save_figure(fig, f"{TARGET}_target_monthly_coverage_timeseries")
    plt.show()

## Lower / upper / range maps — monthly at TARGET_TIME_MONTHLY

Snapshot at one calendar month. The bound is dimensionless [0, 1] per
the per-calendar-month normalization; for the snapshot month, each
HRU's value reflects where this particular year ranks within that
HRU's distribution of that calendar month across all years.

In [ ]:
if ds_monthly_raw is None:
    print("SKIP")
else:
    lb = select_month(ds_monthly_raw["lower_bound"], TARGET_YEAR, TARGET_MONTH).to_pandas()
    ub = select_month(ds_monthly_raw["upper_bound"], TARGET_YEAR, TARGET_MONTH).to_pandas()
    rng = ub - lb
    units = ds_monthly_raw["lower_bound"].attrs.get("units", "1")

    fig, axes = plt.subplots(1, 3, figsize=(22, 7))
    plot_hru_choropleth(axes[0], fabric, lb, vmin=0, vmax=1, cmap="YlGnBu",
        title=f"lower_bound\n{TARGET_TIME_MONTHLY} | {units}", units=units)
    plot_hru_choropleth(axes[1], fabric, ub, vmin=0, vmax=1, cmap="YlGnBu",
        title=f"upper_bound\n{TARGET_TIME_MONTHLY} | {units}", units=units)
    plot_hru_choropleth(axes[2], fabric, rng, vmin=0, vmax=1, cmap="OrRd",
        title=f"range = upper - lower\n{TARGET_TIME_MONTHLY} | {units}", units=units)
    fig.suptitle(f"{TARGET} monthly bounds — {TARGET_TIME_MONTHLY}", fontsize=13, y=1.02)
    plt.tight_layout()
    save_figure(fig, f"{TARGET}_target_monthly_bounds_map")
    plt.show()

    print(f"lower CONUS area-weighted mean: {area_weighted_mean(lb, fabric):.4f}")
    print(f"upper CONUS area-weighted mean: {area_weighted_mean(ub, fabric):.4f}")
    print(f"range CONUS area-weighted mean: {area_weighted_mean(rng, fabric):.4f}")

## Climatological monthly cycle of the bound — monthly variant

The signature plot for the monthly variant: CONUS-mean lower / upper
bound averaged within each calendar month across all years. Two
diagnostics:

- **Seasonal pattern in the spread (upper - lower).** With LSMs as
  the sources, the spread tends to be widest in winter (frozen-soil /
  snowpack regimes confound the schemes) and narrowest in summer.
- **Vertical placement of the climatological mean.** Per-calendar-month
  normalization grand-mean isn't 0.5 (only for symmetric distributions);
  the value reflects the skew of the cross-(HRU, year) distribution
  within each month.

In [ ]:
if ds_monthly_raw is None:
    print("SKIP")
else:
    upper_clim = ds_monthly_raw["upper_bound"].groupby("time.month").mean().mean(id_dim).values
    lower_clim = ds_monthly_raw["lower_bound"].groupby("time.month").mean().mean(id_dim).values
    months = np.arange(1, 13)

    fig, ax = plt.subplots(figsize=(11, 5))
    ax.fill_between(months, lower_clim, upper_clim, color="steelblue", alpha=0.25,
                    label="lower–upper envelope")
    ax.plot(months, lower_clim, color="steelblue", lw=1.5, marker="o", label="lower")
    ax.plot(months, upper_clim, color="darkblue", lw=1.5, marker="o", label="upper")
    ax.set_xticks(months)
    ax.set_xticklabels(["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"])
    ax.set_ylabel("Bound value (dimensionless)")
    ax.set_ylim(0, 1)
    ax.set_title(f"{TARGET} monthly — climatological per-month bounds (CONUS-mean across years)")
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    save_figure(fig, f"{TARGET}_target_monthly_climatology")
    plt.show()

    print("Climatological monthly upper / lower / spread:")
    for m in range(12):
        print(f"  {months[m]:>2}: lower={lower_clim[m]:.4f}  upper={upper_clim[m]:.4f}  spread={upper_clim[m]-lower_clim[m]:.4f}")

## Schema — annual variant

In [ ]:
if ds_annual_raw is None:
    print("SKIP: annual variant not loaded.")
else:
    print(ds_annual_raw)
    print()
    print("=== Global attrs ===")
    for k, v in ds_annual_raw.attrs.items():
        print(f"  {k:<22} {v}")
    print()
    print("=== Per-variable units ===")
    for v in ("lower_bound", "upper_bound", "n_sources"):
        print(f"  {v:<14} units={ds_annual_raw[v].attrs.get('units', '?')!r}  long_name={ds_annual_raw[v].attrs.get('long_name', '')!r}")

## Annual variant — CONUS area-weighted time series

For the annual variant, each year is a single timestep — the CONUS-mean
trace shows the interannual signal directly. Known CONUS climate years
to look for (1982–2010):

- **1983, 1997** — strong El Niño wet years
- **1988** — major Plains/Midwest drought
- **2000, 2002** — late-1990s/early-2000s SW drought peak

The lower / upper envelope should narrow when sources agree and widen
during years with unusual conditions.

In [ ]:
if ds_annual_raw is None:
    print("SKIP")
else:
    lb_series = area_weighted_series(ds_annual_raw["lower_bound"], fabric, id_dim)
    ub_series = area_weighted_series(ds_annual_raw["upper_bound"], fabric, id_dim)
    units = ds_annual_raw["lower_bound"].attrs.get("units", "1")

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.fill_between(lb_series.index, lb_series.values, ub_series.values,
                    color="steelblue", alpha=0.25, label="lower–upper envelope")
    ax.plot(lb_series.index, lb_series.values, color="steelblue", lw=1, marker="o", label="lower")
    ax.plot(lb_series.index, ub_series.values, color="darkblue", lw=1, marker="o", label="upper")
    ax.set_xlabel("Year")
    ax.set_ylabel(f"Area-weighted CONUS mean ({units})")
    ax.set_title(f"{TARGET} annual target — CONUS bounds")
    ax.set_ylim(0, 1)
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    save_figure(fig, f"{TARGET}_target_annual_conus_series")
    plt.show()

    print("CONUS-mean annual lower / upper / spread:")
    for yr, lo, up in zip(pd.DatetimeIndex(lb_series.index).year, lb_series.values, ub_series.values):
        print(f"  {yr}: lower={lo:.4f}  upper={up:.4f}  spread={up-lo:.4f}")

## Annual variant — representative HRU time series

Same four points as monthly; the per-HRU bound here reflects each
year's rank within that HRU's annual distribution across the full
period. Narrow envelopes = sources agree on which years are wet vs
dry at that HRU.

In [ ]:
if ds_annual_raw is None:
    print("SKIP")
else:
    rep_hrus = lookup_hrus_by_points(fabric, REPRESENTATIVE_POINTS)
    lb_at = ds_annual_raw["lower_bound"].sel({id_dim: list(rep_hrus.values())})
    ub_at = ds_annual_raw["upper_bound"].sel({id_dim: list(rep_hrus.values())})

    fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)
    for ax, (label, hru_id) in zip(axes.flat, rep_hrus.items()):
        lb_h = lb_at.sel({id_dim: hru_id}).values
        ub_h = ub_at.sel({id_dim: hru_id}).values
        t = pd.DatetimeIndex(lb_at.time.values)
        ax.fill_between(t, lb_h, ub_h, color="steelblue", alpha=0.25)
        ax.plot(t, lb_h, color="steelblue", lw=1, marker="o", label="lower")
        ax.plot(t, ub_h, color="darkblue", lw=1, marker="o", label="upper")
        ax.set_title(f"{label} (HRU {hru_id})")
        ax.set_ylabel("soil moisture (1)")
        ax.set_ylim(0, 1)
        ax.legend(fontsize=8)
    fig.suptitle(f"{TARGET} annual target at representative HRUs", fontsize=13, y=1.02)
    plt.tight_layout()
    save_figure(fig, f"{TARGET}_target_annual_representative_series")
    plt.show()

## NaN HRU coverage (annual snapshot)

HRUs where `n_sources == 0` for the annual snapshot. Tiny fabric
polygons that miss all source grids show up here. The NN-fill
companion (when present) fills these from the nearest finite HRU.

In [ ]:
if ds_annual_raw is None:
    print("SKIP")
else:
    lb_year = select_month(ds_annual_raw["lower_bound"], TARGET_YEAR, 1).to_pandas()
    fig, ax = plt.subplots(figsize=(11, 7))
    plot_nan_hrus(
        ax, fabric, lb_year,
        title=(
            f"NaN HRUs (red) — {TARGET_TIME_ANNUAL} annual variant | "
            f"{nan_hru_count(lb_year)} of {len(fabric)} "
            f"({100 * nan_hru_count(lb_year) / len(fabric):.2f}%)"
        ),
    )
    plt.tight_layout()
    save_figure(fig, f"{TARGET}_target_annual_nan_map")
    plt.show()

## NN-fill comparison — annual variant

Same NN-fill diagnostic as runoff / AET / recharge: the filled file
should change the CONUS-mean trace minimally (interpolation from
immediate neighbours), and the `nn_filled` flag map should match the
NaN-HRU map one-for-one.

In [ ]:
if ds_annual_raw is None or ds_annual_filled is None:
    print("SKIP: annual NN-filled variant not present.")
else:
    lb_f_series = area_weighted_series(ds_annual_filled["lower_bound"], fabric, id_dim)
    ub_f_series = area_weighted_series(ds_annual_filled["upper_bound"], fabric, id_dim)

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(lb_series.index, lb_series.values, color="steelblue", lw=1, label="lower (raw)")
    ax.plot(lb_f_series.index, lb_f_series.values, color="steelblue", lw=1, ls="--", label="lower (filled)")
    ax.plot(ub_series.index, ub_series.values, color="darkblue", lw=1, label="upper (raw)")
    ax.plot(ub_f_series.index, ub_f_series.values, color="darkblue", lw=1, ls="--", label="upper (filled)")
    ax.set_xlabel("Year")
    ax.set_ylabel("Area-weighted CONUS mean (1)")
    ax.set_title(f"{TARGET} annual — raw vs NN-filled CONUS series")
    ax.set_ylim(0, 1)
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    save_figure(fig, f"{TARGET}_target_annual_nn_fill_series")
    plt.show()

    n_filled_yr = int((ds_annual_filled["nn_filled"].values == 1).sum())
    total = ds_annual_filled["nn_filled"].size
    print(f"NN-filled cells (annual): {n_filled_yr} of {total} ({100*n_filled_yr/total:.4f}%)")

## Bounds-in-[0, 1] invariant check (both variants)

When `normalize_period == period` (default), both bounds must be in
[0, 1] by construction for both variants. When the two differ
(extended-output configurations), values outside the window may be
outside [0, 1] by design.

In [ ]:
for label, ds in (("monthly", ds_monthly_raw), ("annual", ds_annual_raw)):
    if ds is None:
        continue
    lb_arr = ds["lower_bound"].values
    ub_arr = ds["upper_bound"].values
    finite_pair = np.isfinite(lb_arr) & np.isfinite(ub_arr)
    lower_le_upper = bool((ub_arr[finite_pair] >= lb_arr[finite_pair]).all())
    lb_min, lb_max = float(np.nanmin(lb_arr)), float(np.nanmax(lb_arr))
    ub_min, ub_max = float(np.nanmin(ub_arr)), float(np.nanmax(ub_arr))
    period = ds.attrs.get("period", "")
    norm_period = ds.attrs.get("normalize_period", "")
    print(f"--- {label} ---")
    print(f"  lower range: [{lb_min:.4f}, {lb_max:.4f}]")
    print(f"  upper range: [{ub_min:.4f}, {ub_max:.4f}]")
    print(f"  upper >= lower everywhere: {lower_le_upper}")
    if period == norm_period:
        ok = lb_min >= -1e-6 and ub_max <= 1 + 1e-6 and lower_le_upper
        print(f"  period == normalize_period: bounds must be in [0, 1] — {'PASS' if ok else 'FAIL'}")
    else:
        print(f"  period {period!r} != normalize_period {norm_period!r} — bounds outside [0, 1] OK by design")
    print()

In [ ]:
for ds in (ds_monthly_raw, ds_monthly_filled, ds_annual_raw, ds_annual_filled):
    if ds is not None:
        ds.close()